In [0]:
"""
Function_For_Window_And_Overlap_Dataframe_Transformation.py

Python equivalent of Function_For_Window_And_Overlap_Dataframe_Transformation.R

Loads the EDA CSV produced by the earlier conversion script, labels each row
with an experimental condition based on its row index, then reshapes the
data into fixed-size time windows (with optional overlap between condition
blocks) — one row per (participant, window) pair.

Requires: pandas
    pip install pandas
"""

import itertools
import math
import os
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Set working directory to the folder this script lives in
# (equivalent of the rstudioapi current_path / setwd() calls in the R script)
# ---------------------------------------------------------------------------
#current_path = Path(__file__).resolve().parent
#os.chdir(current_path)

df = spark.table("eda.default.eda_bronze_raw").toPandas()

# ---------------------------------------------------------------------------
# Function to assign condition based on row index (1-based, matching R)
# ---------------------------------------------------------------------------
markers = [
    60,           # end of n
    2 * 60 + 15,  # end of b
    3 * 60 + 15,  # end of p
    4 * 60 + 29,  # end of b
    5 * 60 + 26,  # end of negative
]
markers = [m * 4 for m in markers]


def assign_condition_by_row(row_index):
    """row_index is 1-based, matching R's indexing."""
    if 1 + 240 <= row_index <= markers[0] - 1 + 240:
        return "neutral"
    if markers[0] + 240 <= row_index <= markers[1] - 1 + 240:
        return "black"
    if markers[1] + 240 <= row_index <= markers[2] - 1 + 240:
        return "positive"
    if markers[2] + 240 <= row_index <= markers[3] - 1 + 240:
        return "black"
    if row_index >= markers[3] + 240:
        return "negative"
    return None


df["condition"] = [assign_condition_by_row(i) for i in range(1, len(df) + 1)]

df = df[df["condition"].notna()].reset_index(drop=True)

# Convert to an ordered categorical (equivalent of R's factor with levels)
condition_levels = ["neutral", "positive", "negative", "black"]
df["condition"] = pd.Categorical(df["condition"], categories=condition_levels)

# All column names excluding 'condition'
colnames = [c for c in df.columns if c != "condition"]


# ---------------------------------------------------------------------------
# Reshape function
# ---------------------------------------------------------------------------
def reshape_by_condition(df, window_size=10, overlap=0):
    chunk_size = window_size * 4

    colnames_data = [c for c in df.columns if c != "condition"]
    conditions = df["condition"].astype(str).tolist()

    # --- run-length encoding of the condition column -----------------------
    values = []
    lengths = []
    for val, grp in itertools.groupby(conditions):
        values.append(val)
        lengths.append(len(list(grp)))

    block_ends_count = list(itertools.accumulate(lengths))  # cumulative counts
    block_starts = [0] + block_ends_count[:-1]  # 0-based start index of each block

    # --- build chunks (windows) per condition block -------------------------
    chunk_list = []
    for start, end_count, cond in zip(block_starts, block_ends_count, values):
        end = end_count - 1  # 0-based inclusive end index
        row_indices = list(range(start, end + 1))
        n_rows = len(row_indices)
        n_chunks = math.ceil(n_rows / chunk_size)

        padded = row_indices + [None] * (n_chunks * chunk_size - n_rows)
        chunks = [padded[j * chunk_size:(j + 1) * chunk_size] for j in range(n_chunks)]

        chunk_list.append({"chunks": chunks, "condition": cond, "n_chunks": n_chunks})

    # --- handle overlap between condition blocks -----------------------------
    if overlap > 0:
        for i in range(len(chunk_list) - 1):
            current_block = chunk_list[i]
            next_block_start = block_starts[i + 1]
            next_block_end = block_ends_count[i + 1] - 1  # 0-based inclusive

            overlap_end = min(next_block_start + overlap - 1, next_block_end)
            overlap_indices = (
                list(range(next_block_start, overlap_end + 1))
                if overlap_end >= next_block_start
                else []
            )

            last_chunk = current_block["chunks"][-1]
            real_indices = [x for x in last_chunk if x is not None]

            combined = real_indices + overlap_indices

            if len(combined) <= chunk_size:
                last_chunk_new = combined + [None] * (chunk_size - len(combined))
            else:
                last_chunk_new = combined

            chunk_list[i]["chunks"][-1] = last_chunk_new

    # --- flatten all chunks across blocks -------------------------------------
    all_chunks = []
    all_conditions = []
    for block in chunk_list:
        all_chunks.extend(block["chunks"])
        all_conditions.extend([block["condition"]] * block["n_chunks"])

    n_cols2 = len(all_chunks)

    # --- build one row per (column_name, chunk) pair --------------------------
    rows = []
    row_labels = []

    for col_name in colnames_data:
        vals = df[col_name].to_numpy()
        for chunk_idx, idx_list in enumerate(all_chunks):
            out = [None] * max(chunk_size, len(idx_list))
            for pos, idx in enumerate(idx_list):
                if idx is not None:
                    out[pos] = vals[idx]
            out = out[:chunk_size]

            rows.append(out)
            label = f"{col_name}_{(chunk_idx + 1) * window_size}s"
            row_labels.append(label)

    value_colnames = [f"t{j + 1}" for j in range(chunk_size)]
    df_reshaped = pd.DataFrame(rows, index=row_labels, columns=value_colnames)

    # condition / source_rows recycle across each column_name's block of rows
    df_reshaped["condition"] = pd.Categorical(
        all_conditions * len(colnames_data), categories=condition_levels
    )

    source_rows_str = [
        ", ".join(str(idx + 1) for idx in idx_list if idx is not None)
        for idx_list in all_chunks
    ]
    df_reshaped["source_rows"] = source_rows_str * len(colnames_data)

    return df_reshaped


# ---------------------------------------------------------------------------
# Default: window = 10, no overlap
# ---------------------------------------------------------------------------
result = reshape_by_condition(df)
result.to_csv("../../Data/Processed/Window_10_No_Overlap.csv", index=True)

# Save the reshaped dataset out to a Silver Delta Table
spark_result = spark.createDataFrame(result.reset_index()) # preserve the index strings
spark_result.write.format("delta").mode("overwrite").saveAsTable("eda.default.eda_silver_windowed")

# Window of 20, no overlap
result = reshape_by_condition(df, window_size=20)

# Window of 10, allow 2 rows of overlap at condition boundaries
result = reshape_by_condition(df, window_size=10, overlap=2)

# Both changed
result = reshape_by_condition(df, window_size=20, overlap=5)